### Import Config

In [2]:
import json
import glob
import pandas as pd
from pathlib import Path

## Parse the JSON

In [ ]:
# Find all the json files
files = glob.glob('../data/api_raw/products/*.json')

# Iterate and parse   
rows = []

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    product = data.get('product', {})

    # ASIN
    asin = Path(file).stem
    
    # Categories
    categories = product.get('categories', [])

    # Buybox
    buybox = product.get('buybox_winner', {})
    fulfillment = buybox.get('fulfillment', {})
    seller = fulfillment.get('third_party_seller', {})

    # Best Sellers Rank
    bsr = product.get('bestsellers_rank', [])

    rows.append({
        'asin':        asin,
        'brand':       product.get('brand'),
        'subcategory': categories[-1].get('name') if categories else None,
        'image':       product.get('main_image', {}).get('link'),
        'is_fba':      fulfillment.get('is_fulfilled_by_amazon'),
        'bsr_main':    bsr[0].get('rank') if len(bsr) > 0 else None,
        'bsr_sub':     bsr[1].get('rank') if len(bsr) > 1 else None,
        'seller_name': seller.get('name')
    })


### Create Products DataFrame 

In [7]:
df_products = pd.DataFrame(rows)
df_products.head(10)

,asin,brand,subcategory,image,is_fba,bsr_main,bsr_sub,seller_name
0,B0009KF59W,‎WILSON,Basketballs,https://m.media-amazon.com/images/I/81ATTfCvMI...,True,90.0,1.0,None
1,B00AHAWWO0,Crest,Strips,https://m.media-amazon.com/images/I/715rFhZpV0...,True,121.0,1.0,None
2,B0113UZJE2,‎Etekcity,Digital Scales,https://m.media-amazon.com/images/I/91YrLTBnMc...,True,3.0,1.0,None
3,B01FWAZEIU,APC,Uninterruptible Power Supply (UPS),https://m.media-amazon.com/images/I/61-O0p1npO...,True,1.0,NaN,None
4,B07HJSHT8P,‎Franklin Sports,Batting Tees,https://m.media-amazon.com/images/I/61-axin60N...,True,107.0,1.0,None
5,B07HRCDDL1,MR.SIGA,Cleaning Cloths,https://m.media-amazon.com/images/I/91AFRzSSLb...,True,395.0,2.0,Mr SIGA USA
6,B07PZF3QS3,‎KitchenAid,Shears,https://m.media-amazon.com/images/I/51Byq+vTy1...,True,14.0,1.0,None
7,B088H5DWV3,WEERTI,Sets,https://m.media-amazon.com/images/I/51T4Bf02rP...,True,922.0,1.0,WEERTI
8,B08JGNRJ8P,‎BooTaa,Dartboards,https://m.media-amazon.com/images/I/71CRKRGYHC...,True,846.0,1.0,None
9,B08KT2Z93D,eos,Lotions,https://m.media-amazon.com/images/I/51lP01--ej...,True,2.0,1.0,None


### Create CSV Files

In [8]:
df_products.to_csv('../data/processed/products_enriched.csv', index=False)
print(f"Saved: {df_products.shape}")

Saved: (32, 8)
